# Post-Processing: Recalculating Statuses After Scoring

A post-processor runs after scoring and may change field statuses (match, mismatch, omission, hallucination,skipped), and thus influence the score. There is a built-in post-processor `reclassify_nulls`, it changes **what counts as an omission or a hallucination**.

There are two ways an extractor deifnes omission and hallucination. Please check `docs/scoring.md` for more details.

**1. The key is missing.** This is the default definition.

```
gold       {"method": "PVD", "temp": 300}
extracted  {"method": "PVD"}                 ->  temp: omission

gold       {"method": "PVD"}
extracted  {"method": "PVD", "temp": 300}    ->  temp: hallucination
```

**2. The key is there, but the value is empty** (`null`, `""`, ...). Default scoring sees a key
on both sides and calls it a mismatch. `reclassify_nulls` treats the empty value as absent, so
it becomes an omission or hallucination too.

```
gold       {"method": "PVD", "temp": 300}
extracted  {"method": "PVD", "temp": null}   ->  default: mismatch     can use reclassify_nulls to make it omission

gold       {"method": "PVD", "temp": null}
extracted  {"method": "PVD", "temp": 300}    ->  default: mismatch     can use reclassify_nulls to make it hallucination

gold       {"method": "PVD", "temp": null}
extracted  {"method": "PVD", "temp": null}   ->  default: match        can use reclassify_nulls to make it skipped 
```


## Why the second definition is needed

**Constrained-output tools** (Outlines, Instructor, OpenAI structured outputs) force every
schema key to appear in the output. The model cannot leave a key out; it says "I don't know"
with `null` or `""`. If gold is produced with the same schema, every key exists on both sides,
so under the default definition there are **no omissions and no hallucinations**: everything
is a match or a mismatch, and precision equals recall. "Wrong answer" and "no answer" become
indistinguishable.

`reclassify_nulls` restores the distinction by treating configured values as absent:

| Gold | Extracted | Default | With `reclassify_nulls` |
|------|-----------|---------|-------------------------|
| value | value | match / mismatch | unchanged |
| value | absent | mismatch | **omission** (extractor gave up) |
| absent | value | mismatch | **hallucination** (extractor invented) |
| absent | absent | match | **skipped** or match (configurable) |


## Example Data

In [10]:
GOLD = [
    # Record 0: gold has all fields
    {"method": "PVD", "temp": 300, "notes": "Substrate pre-heated."},
    # Record 1: gold has no notes value (null)
    {"method": "CVD", "temp": 500, "notes": None},
]

EXTRACTED = [
    # Extractor gave up on temp (null) and notes ("")
    {"method": "PVD", "temp": None, "notes": ""},
    # Extractor invented notes that don't exist in gold
    {"method": "CVD", "temp": 500, "notes": "N/A"},
]

## Step 1: Default Scoring (no post-processing)

`null` and `""` are treated as regular values.

In [14]:
from struct_extract_eval import evaluate, infer_schema, annotate_xeval
from example_utils import show_run

schema = infer_schema(GOLD)
annotate_xeval(schema)

run_default = evaluate(GOLD, EXTRACTED, schema=schema)
show_run(run_default, "Default scoring (null is a value):")


Default scoring (null is a value):
  mean P=0.50  R=0.50  F1=0.50   (2 record(s))
  record  path    gold                     extracted  score  status    reason
  0       method  'PVD'                    'PVD'      1.0    match
  0       notes   'Substrate pre-heated.'  ''         0.0    mismatch  mismatch
  0       temp    300                      None       0.0    mismatch  type_error
  1       method  'CVD'                    'CVD'      1.0    match
  1       notes   None                     'N/A'      0.0    mismatch  mismatch
  1       temp    500                      500        1.0    match


Everything is a match or mismatch. No omissions, no hallucinations. Precision == Recall.

## Step 2: With Null Handling

Configure which values mean "absent" and pass `reclassify_nulls` as a post-processor.

In [16]:
from struct_extract_eval.postprocess import NullHandling, reclassify_nulls

config = NullHandling(
    absent_values=[None, ""],  # both null and empty string mean "no answer"
    both_absent_skip=True,     # default true, null vs null = skip (excluded from metrics), otherwise, null vs null = match (score 1.0)
)

run_null = evaluate(
    GOLD, EXTRACTED, schema=schema,
    post_process=lambda frs: reclassify_nulls(frs, config),
)

show_run(run_null, "With null handling (null/empty = absent):")

With null handling (null/empty = absent):
  mean P=0.83  R=0.67  F1=0.65   (2 record(s))
  record  path    gold                     extracted  score  status         reason
  0       method  'PVD'                    'PVD'      1.0    match
  0       notes   'Substrate pre-heated.'  ''         0.0    omission       extracted is absent
  0       temp    300                      None       0.0    omission       extracted is absent
  1       method  'CVD'                    'CVD'      1.0    match
  1       notes   None                     'N/A'      0.0    hallucination  gold is absent
  1       temp    500                      500        1.0    match


Result:
- **Record 0, `temp`:** gold=300, extracted=null -> **omission** (extractor gave up). Hurts recall.
- **Record 0, `notes`:** gold="Substrate pre-heated.", extracted="" -> **omission**. Hurts recall.
- **Record 1, `notes`:** gold=null, extracted="N/A" -> **hallucination** (extractor invented). Hurts precision. If your extractor uses "N/A" to mean "no answer" (same as null), add it to `absent_values`: `NullHandling(absent_values=[None, "", "N/A"])`, and if you further set `both_absent_skip=False`, `None` will match `N/A`.


## Configuration Options

### `absent_values`

Which values count as "absent." All values in the list are treated as equivalent:

```python
NullHandling(absent_values=[None])          # only null
NullHandling(absent_values=[None, ""])      # null or empty string
NullHandling(absent_values=[None, "", []])  # null, empty string, or empty list
```

This means `None` vs `""` = both absent (same category), not a mismatch.

### `both_absent_skip`

What happens when both gold and extracted are absent:

```python
both_absent_skip=True   # skip -- excluded from metrics entirely (default)
both_absent_skip=False  # match -- counts as a correct extraction (score 1.0)
```

## Writing Your Own Post-Processor

`post_process` accepts any function `(list[FieldResult]) -> list[FieldResult]`.
You can chain multiple post-processors:

```python
from struct_extract_eval.postprocess import reclassify_nulls, propagate_batch_errors

def my_post_process_chain(frs):
    # If the LLM judge (batch comparator) failed on any field in a record,
    # its other "successful" results in the same batch may be unreliable
    # (e.g. the judge miscounted or misaligned responses). This marks ALL
    # fields from that tainted batch as batch_error so they don't pollute
    # metrics with potentially wrong scores.
    propagate_batch_errors(frs)
    reclassify_nulls(frs, config)
    my_post_process(frs)
    return frs

result = evaluate(gold, extracted, schema, post_process=my_post_process)
```

Post-processors run **after** batch comparator dispatch. This package includes two
built-in post-processors: `reclassify_nulls` (null handling) and
`propagate_batch_errors` (batch error handling). You can also write and apply your own.